In [57]:
import string
import random
import os
os.makedirs("test_output", exist_ok = True)
alphabet = string.ascii_lowercase

In [58]:
def msg_prep(message):
    return "".join(ch for ch in message if ch in string.ascii_letters)

# Function to shift a character by a given amount
def shift_char(ch,shift):
    if ch.islower():
        base = ord("a")     # ord("a") finds the ASCII number for "a" 
    else:
        base = ord("A")     # If the letter is uppercase we take the ASCII for "A"
    return chr((ord(ch) - base + shift) % 26 + base)  # Returns the shifted letter. % 26 because we need the alphabet to wrap around

# Caesar cypher encryption and decryption
def caesar_encrypt(message, shift):
    prepped_msg = msg_prep(message)
    if not 0 <= shift <= 25:        # A shift of 26 is just a shift of 0
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, shift) for ch in prepped_msg)     # Shifts each character in prepped_msg and joins them with no spaces

def caesar_decrypt(ciphertext, shift):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not 0 <= shift <= 25:
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, -shift) for ch in ciphertext) # -shift since we are going back to what it was, so subtract the shift.

print(caesar_encrypt("hellohowareyou", 20))
print(caesar_decrypt(caesar_encrypt("hellohowareyou", 20), 20))

byffibiqulysio
hellohowareyou


In [59]:
# Function to convert the characters in the key to individual shifts
def key_to_shifts(key):
    keylist = []
    for ch in key:
        keylist.append(ord(ch.lower()) - ord("a"))  # ch.lower makes ch lowercase so that "KEY" and "key" have the same output
    return keylist

def vigenere_encrypt(message, key):
    prepped_msg = msg_prep(message)
    if not key or not key.isalpha():    # The key needs to be non-empty and be only alphabetical characters
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(prepped_msg[i], shifts[i % len(key)]) for i in range(len(prepped_msg)))     # Shifts each character from prepped_msg with the correct shift acquired from the key and joins them

def vigenere_decrypt(ciphertext, key):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not key or not key.isalpha():
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(ciphertext[i], -shifts[i % len(key)]) for i in range(len(ciphertext)))      # -shifts because we need to shift backwards to the original text

print(vigenere_encrypt("hello", "key"))
print(vigenere_decrypt(vigenere_encrypt("hello", "key"), "key"))

rijvs
hello


In [60]:
# Brief asks that encryption and decryption be implemented to and from files

# Strips everything but letters in a text
def extract_text(text):
    return "".join(ch for ch in text if ch in string.ascii_letters)

# Opens a file and returns the text in it as a string
def read_file(path):
    with open(path, "r", encoding = "utf-8") as f:
        return f.read()

# Saves a string to a file
def write_file(path, text):
    with open(path, "w", encoding = "utf-8") as f:
        f.write(text)

# Define functions which perform the two forms of encryption and decryption on text files
def caesar_encrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_encrypt(extract_text(file_to_string), shift))

def caesar_decrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_decrypt(file_to_string, shift))

def vigenere_encrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_encrypt(extract_text(file_to_string), key))

def vigenere_decrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_decrypt(file_to_string, key))

In [61]:
# Random shifts for Caesar cypher testing
def random_caesar_shift():
    return random.randint(0,25)

# Random keys for Vigenere cypher testing
def random_vigenere_keys(length):
    return "".join(random.choice(alphabet) for _ in range(length))

# Returns a random message made up of lower and uppercase letters of a specified length
def random_message(length):
    return "".join(random.choice(string.ascii_letters) for _ in range(length))

def test_random(trials = 100):
    for _ in range(trials):
        r_msg = random_message(random.randint(1, 200))
        r_shift = random_caesar_shift()
        r_key = random_vigenere_keys(random.randint(1, len(r_msg)))
        assert caesar_decrypt(caesar_encrypt(r_msg, r_shift), r_shift) == r_msg
        assert vigenere_decrypt(vigenere_encrypt(r_msg, r_key), r_key) == r_msg
        assert vigenere_encrypt(r_msg, alphabet[r_shift]) == caesar_encrypt(r_msg, r_shift)   # A single-letter key should behave like the Caesar cypher
    print(f"Random tests: passed {trials} trials.")


def test_caesar(message):
    for shift in range(26):
        assert caesar_decrypt(caesar_encrypt(message, shift), shift) == message
    print(f"Caesar: Passed all 26 shifts.")

test_caesar("HelloWorld")
test_random()

def test_files(file):
    original = extract_text(read_file(file))
    r_shift = random_caesar_shift()
    caesar_encrypt_file(file, "test_output/caesar_encrypted.txt", r_shift)
    caesar_decrypt_file("test_output/caesar_encrypted.txt", "test_output/caesar_decrypted.txt", r_shift)
    assert read_file("test_output/caesar_decrypted.txt") == original

    r_key = random_vigenere_keys(random.randint(1, len(original)))
    vigenere_encrypt_file(file, "test_output/vigenere_encrypted.txt", r_key)
    vigenere_decrypt_file("test_output/vigenere_encrypted.txt", "test_output/vigenere_decrypted.txt", r_key)
    assert read_file("test_output/vigenere_decrypted.txt") == original
    print(f"File tests passed for {file}")

test_files("message_moby_dick.txt")
test_files("message_wuthering_heights.txt")

Caesar: Passed all 26 shifts.
Random tests: passed 100 trials.
File tests passed for message_moby_dick.txt
File tests passed for message_wuthering_heights.txt


In [62]:
from cryptography_functions import *

# Functions from lectures that weren't in cryptography_functions.py
def RSA_encrypt(message, N, e):
    return pow(message, e, N)

def RSA_decrypt(ciphertext, N, d):
    return pow(ciphertext, d, N)

In [ ]:
# Alice generates a key using the RSA_key_generator function from the lectures
def alices_key():
    return RSA_key_generator()

# key_slicer takes a string as input and returns strings of at most length size in a list. The last element in the list may be shorter
def key_slicer(key, size):
    slices = []
    for i in range(0, len(key), size):  # The 'i's range from 0 to len(key) with jumps of value size. So 0, size, 2*size, ... etc until len(key) is reached
       slices.append(key[i : i + size])
    return slices

def bobs_cipher(message, N, e):
    bobs_key = random_vigenere_keys(random.randint(1, len(message)))
    bobs_slices = key_slicer(bobs_key, (N.bit_length() - 1) // 8)   # Here size == (N.bit_length() - 1) // 8 since we must have that the integers being RSA encrypted are less than N, and a slice of this size guarantees it
    encrypted_key_integers = []
    for key_slice in bobs_slices:
        key_integer = convert_to_integer(key_slice)
        encrypted_key_integers.append(RSA_encrypt(key_integer, N, e))
    ciphered_message = vigenere_encrypt(message, bobs_key)
    return ciphered_message, tuple(encrypted_key_integers)      # Brief asks for ciphertext integers to be transmitted as a tuple

def alices_decypher(ciphertext, encrypted_key_integers, N, d):
    deciphered_key_slices = []
    for encrypted_key_slice in encrypted_key_integers:
        decrypted_key_integers = RSA_decrypt(encrypted_key_slice, N, d)
        deciphered_key_slices.append(convert_to_text(decrypted_key_integers))
    vig_key = "".join(deciphered_key_slices)
    bobs_message = vigenere_decrypt(ciphertext, vig_key)
    return bobs_message

In [67]:
# Basic test
(N,e), d, factors = alices_key()
ciphertext, ciphered_key = bobs_cipher("text with capitals, spaces, commas, a question mark AND an exclamation mark?!", N, e)
print(f"({ciphertext}, {ciphered_key})")
print(alices_decypher(ciphertext, ciphered_key, N, d))


(hhfqkrimcoaxobjjlzrchhbsdtmvfycgvqqlaqgtyDVAowtcczlbvugfgwrrn, (11456557173033795081264605797601872490584696295757203481714705504528768628060, 5112540637644645416630024747682328676328995354540694464903715963063628112663))
textwithcapitalsspacescommasaquestionmarkANDanexclamationmark
